# Notebook 01 — The full pipeline, from cross sections to LET

This is the **first notebook to read**. It walks through the whole chain of
an event-by-event track-structure simulation, one step at a time, on
single protons in liquid water:

| § | Step | Question it answers |
|---|---|---|
| 1 | Load the cross sections | How likely is each collision process at energy $E$? |
| 2 | Free-path sampling | Does the code sample the distance between collisions correctly? |
| 3 | One track | What does the code produce for one proton, and is energy conserved? |
| 4 | Stopping power: Monte Carlo vs analytic | Does the sampling reproduce the exact expectation of its own rules? |
| 5 | Stopping power vs NIST PSTAR | How does the model compare with the standard reference data? |
| 6 | Stopping power vs experiment | ...and with measurements? Why do the gaps look the way they do? |
| 7 | Charge-state equilibrium | Does the sampled H⁺ ↔ H⁰ switching give the expected charge fractions? |
| 8 | Bragg curve | How is dose distributed with depth along one track? |
| 9 | LET | What are the track- and dose-averaged LET, and why does $\bar L_D$ need a length scale? |

**Run time: about 4 minutes.** §5 and §6 take most of it. All figures are
saved in `notebooks/output_01/` as PNG and PDF.

**Units.** Energies in eV, lengths in nm, cross sections in nm², stopping
power and LET in eV/nm, which is numerically equal to keV/µm. For hydrogen,
energies in keV/u (per nucleon) equal keV.

**What the model includes, and what it does not.** All five processes
(ionization, excitation, electron capture, electron loss, elastic
scattering) use real tabulated cross sections (CDW-EIS, Miller–Green and
classical models) and energy-dependent mean energy losses. δ-electrons
deposit their energy where they are created; they are not transported.
§5–6 compare the result with reference data; the code has not been
tuned to that data. See the main README for the full list of
limitations.

In [ ]:
# Google Colab only: download the repository and install the package.
# On your own computer (after ./setup.sh) this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/ion-track-mc"):
        subprocess.run(["git", "clone", "-q", "https://github.com/juanmontiz/ion-track-mc",
                        "/content/ion-track-mc"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/content/ion-track-mc"], check=True)
    os.chdir("/content/ion-track-mc/notebooks")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from ion_track_mc import constants
from ion_track_mc.physics.channels import Channel
from ion_track_mc.physics.cross_section import (
    TabulatedCrossSectionSet,
    load_cross_section_table,
    load_mean_energy_table,
)
from ion_track_mc.physics.medium import WATER
from ion_track_mc.transport import sampler
from ion_track_mc.transport.engine import TrackEngine
from ion_track_mc.transport.particle import Particle
from ion_track_mc.analysis.stopping_power import (
    analytic_equilibrium_stopping_power, analytic_stopping_power,
    emergent_equilibrium_stopping_power, emergent_stopping_power,
    equilibrium_charge_fractions, sample_equilibrium_charge_fractions,
    sample_equilibrium_stopping_power,
)
from ion_track_mc.analysis.visualization import (
    plot_cross_sections, plot_mean_free_path, plot_stopping_power_self_consistency,
    plot_bragg_curve, use_paper_style, save_figure,
)
from ion_track_mc.scoring.dose import make_bragg_curve_scorer
from ion_track_mc.scoring.radiobiology import track_averaged_LET, dose_averaged_LET

use_paper_style()

OUT_DIR = Path("output_01")  # every figure below is also saved here as PNG + PDF

### Monte Carlo sample sizes

Every stochastic estimate in this notebook uses a finite number of random
samples. More samples mean less statistical noise and a longer run. All
sample sizes are collected in the next cell, so you can change them in one
place: halving them roughly halves the run time and increases the noise by
about $\sqrt 2$.

In [ ]:
N_FREE_PATH_SAMPLES = 20000              # §2: free-path sampling vs analytic lambda(E)

N_SELFCONSISTENCY_ENERGY_POINTS = 20     # §4: self-consistency-only energy grid (independent of §5-6's grid)
N_SELFCONSISTENCY_SAMPLES_PER_POINT = 6000

N_SP_ENERGY_POINTS = 30                  # §5-6: shared energy grid -- §6 reuses §5's E_grid/S_eff directly, so these two sections must use the same value
N_SP_SAMPLES_PER_POINT = 6000            # §5-6: MC samples per energy point (self-consistency curve and the blended-fixed-q equilibrium campaign both use this)

N_CHAIN_COLLISIONS = 30000               # §5, §7: stationary-mode collision-chain length (charge-equilibrium fractions / fully emergent equilibrium stopping power)

## 1. Load the cross sections

A **cross section** $\sigma_c(E)$ [nm²] is the effective target area one
water molecule presents to the projectile for process $c$ at energy $E$.
Multiplied by the number of molecules per nm³, $N = 33.4\ \mathrm{nm^{-3}}$
for water, it gives $N\sigma_c$, the number of collisions of type $c$ per
nm of path.

The projectile can be a proton (H⁺, `q = 1`) or a neutral hydrogen atom
(H⁰, `q = 0`), and each has its own set of processes:

- H⁺: ionization, excitation, **capture** (H⁺ → H⁰), elastic;
- H⁰: ionization, excitation, **loss** (H⁰ → H⁺), elastic.

That is why the tables are keyed by `(q, channel)`. The cell also loads the
**mean-energy tables**, which set how much each collision costs:
`Edepmeantot` is the energy left at the collision site, and `Eemeantot` is
the kinetic energy of the δ-electron ejected by an ionization. See
`ion_track_mc/data/README_data_format.md` for what each file holds.

The printed energy range, 8 keV to 300 MeV, is where the tables are
valid. The code never extrapolates outside it.

In [ ]:
DATA_DIR = Path("..") / "ion_track_mc" / "data" / "cross_sections"
FILES = {
    (1, Channel.IONIZATION): "H1+_+_H2O_ionization_TCS_CDW-EIS.dat",
    (1, Channel.EXCITATION): "H1+_+_H2O_excitation_TCS_Miller-Green.dat",
    (1, Channel.CAPTURE): "H1+_+_H2O_capture_TCS_CDW-EIS.dat",
    (1, Channel.ELASTIC): "H1+_+_H2O_elastic_TCS_Classical.dat",
    (0, Channel.IONIZATION): "H0_+_H2O_ionization_TCS_CDW-EIS.dat",
    (0, Channel.EXCITATION): "H0_+_H2O_excitation_TCS_Miller-Green.dat",
    (0, Channel.LOSS): "H0_+_H2O_loss_TCS_Miller-Green.dat",
    (0, Channel.ELASTIC): "H0_+_H2O_elastic_TCS_Classical.dat",
}
tables = {k: load_cross_section_table(DATA_DIR / f) for k, f in FILES.items()}
xs = TabulatedCrossSectionSet(tables)
print("q=1 channels:", xs.channels(1))
print("q=0 channels:", xs.channels(0))
print("q=1 energy range [eV]:", xs.energy_range(1))

EDEP_FILES = {
    (1, Channel.EXCITATION): "H1+_+_H2O_excitation_Edepmeantot_Miller-Green.dat",
    (1, Channel.IONIZATION): "H1+_+_H2O_ionization_Edepmeantot_CDW-EIS.dat",
    (1, Channel.CAPTURE):    "H1+_+_H2O_capture_Edepmeantot_CDW-EIS.dat",
    (0, Channel.EXCITATION): "H0_+_H2O_excitation_Edepmeantot_Miller-Green.dat",
    (0, Channel.IONIZATION): "H0_+_H2O_ionization_Edepmeantot_CDW-EIS.dat",
    (0, Channel.LOSS):       "H0_+_H2O_loss_Edepmeantot_constant.dat",
}
EEMEAN_FILES = {
    1: "H1+_+_H2O_ionization_Eemeantot_CDW-EIS.dat",
    0: "H0_+_H2O_ionization_Eemeantot_CDW-EIS.dat",
}
edep_tables = {k: load_mean_energy_table(DATA_DIR / f) for k, f in EDEP_FILES.items()}
eemean_tables = {k: load_mean_energy_table(DATA_DIR / f) for k, f in EEMEAN_FILES.items()}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
plot_cross_sections(xs, q=1, ax=axes[0], line_width=1.5)
plot_mean_free_path(xs, WATER, q=1, ax=axes[1], line_width=1.5)
fig.tight_layout()
save_figure(fig, OUT_DIR, "01_cross_sections_mean_free_path")

**How to read the figure.**

- **Left: cross sections for H⁺.** At high energy, ionization (red)
  dominates. Below about 37 keV, **electron capture** (blue) becomes the
  largest process: a slow proton easily picks up an electron. Capture also
  falls by more than 15 orders of magnitude from 8 keV to 300 MeV, so at
  clinical energies it is irrelevant except at the very end of the track.
- **Right: mean free path** $\lambda(E) = 1/(N\sigma_{\text{tot}})$, the
  average distance between two collisions. It grows from about 0.1 nm at
  8 keV to about 150 nm at 300 MeV. Slow protons collide far more often.
  That alone does not make the energy loss per nm larger, since each
  collision also costs less at low energy; the energy lost per nm is the
  stopping power, §4.

## 2. Free-path sampling vs the analytic exponential law

Between collisions the proton flies in a straight line. The probability of
flying a distance $s$ without colliding is $e^{-s/\lambda}$, so the
distance to the first collision has the density

$$p(s) = \frac{1}{\lambda}\,e^{-s/\lambda}, \qquad \lambda = \frac{1}{N\sigma_{\text{tot}}(E)} .$$

The code draws $s = -\lambda \ln \xi$ with $\xi$ uniform in $(0,1)$
(`sampler.sample_step`); this is inverse-transform sampling of $p(s)$.
Everything else rests on this step, so it is checked first: the cell
samples 20 000 steps at 500 keV/u and compares their histogram with $p(s)$.
A full statistical test (Kolmogorov–Smirnov) is in `tests/test_sampler.py`.

In [ ]:
E_probe = 500.0e3 * constants.PROTON_NUCLEON_COUNT  # 500 keV/u
rng = np.random.default_rng(0)
steps = np.array([sampler.sample_step(E_probe, 1, WATER.N, xs, rng) for _ in range(N_FREE_PATH_SAMPLES)])
mean_free_path = 1.0 / (WATER.N * xs.sigma_total(E_probe, 1))

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.hist(steps, bins=60, density=True, alpha=0.6, label="MC sampled steps")
s_grid = np.linspace(0, steps.max(), 200)
ax.plot(s_grid, np.exp(-s_grid / mean_free_path) / mean_free_path, "r-", label="Analytic exponential")
ax.set_xlabel("Step length [nm]")
ax.set_ylabel("Probability density")
ax.legend()
print(f"empirical mean = {steps.mean():.4f} nm, analytic lambda = {mean_free_path:.4f} nm")
save_figure(fig, OUT_DIR, "02_free_path_histogram")

**Reading the result.** The histogram follows the red exponential, and
the sample mean (0.879 nm) agrees with the analytic $\lambda = 0.885$ nm to
within the statistical uncertainty of 20 000 samples ($\lambda/\sqrt{N}
\approx 0.006$ nm). At 500 keV/u a proton collides about once per nm.

## 3. Transport one proton track

Now a whole track. `TrackEngine` is the event-by-event loop: sample the
step, move, choose the process with probability
$\sigma_c/\sigma_{\text{tot}}$, apply it, and repeat until the energy falls
below 8 keV (see `ion_track_mc/transport/README_transport.md`). One call to
`transport()` runs the full track of one 500 keV/u proton, including all
the δ-electrons it ejects.

The second printed line is an **energy-conservation check**: the energy of
all recorded deposits must equal the energy the proton lost.

In [ ]:
engine = TrackEngine(WATER, xs, edep_tables=edep_tables, eemean_tables=eemean_tables, seed=7)
primary = Particle(kind="proton", position=np.zeros(3), direction=np.array([0.0, 0.0, 1.0]),
                    energy=E_probe, q=1)
e0 = primary.energy
spent = engine.transport(primary)

print(f"started at {e0:.1f} eV, ended at {spent.energy:.1f} eV")
print(f"total deposits: {len(engine.deposits)}, total energy accounted: "
      f"{sum(d.energy for d in engine.deposits):.1f} eV (should equal {e0 - spent.energy:.1f} eV)")

**Reading the result.**

- The proton lost $500 - 7.98 = 492.0$ keV in 24 444 energy deposits
  (proton collisions plus the δ-electron deposits). On average that is
  about 20 eV per deposit.
- The deposits add up to exactly the energy lost: energy is conserved.
- The track ended at 7.98 keV. That is just below the 8 keV table floor,
  where the code stops following the proton. The last ~8 keV are not
  deposited; this is a small loss compared with 500 keV.

### The stopping-power curves of §4–§6

**Stopping power** $S(E)$ is the mean energy a projectile loses per unit
path length at energy $E$, in eV/nm. It sets the range of the ion and the
shape of the Bragg curve, so it is the first thing to check in any
transport code. Three versions of $S(E)$ computed by this code appear
below:

- **"Analytic"** (blue line). `analytic_stopping_power(..., q=1)`, the
  exact expectation of the code's own rules for a projectile that stays H⁺:
  $$S(E) = N \sum_c \sigma_c(E)\,\langle \Delta E_c \rangle ,$$
  i.e. (collisions of type $c$ per nm) × (mean energy each one costs),
  summed over processes. There is no sampling noise. *In the figures, the
  legend for this curve reads "Analytic (design doc §4.4)"; it is this
  formula.*
- **"Monte Carlo (emergent)"** (orange dots). `emergent_stopping_power`
  draws thousands of random collisions at fixed $E$ and computes
  $\sum \Delta E_i / \sum s_i$. It should agree with the analytic curve.
  This tests the sampling, not the physics.
- **"Analytic, charge-equilibrium blended"** (green).
  `analytic_equilibrium_stopping_power`. A real "proton" beam keeps
  capturing and losing electrons, so at each energy it spends a fraction
  $f_{H^+}$ of its path as H⁺ and $f_{H^0}$ as H⁰. The two fractions follow
  from balancing capture against loss:
  $$f_{H^+} = \frac{\sigma_L}{\sigma_L+\sigma_C}, \qquad f_{H^0} = \frac{\sigma_C}{\sigma_L+\sigma_C},$$
  and the stopping power to compare with measurements is the blend
  $$S(E) = f_{H^+}\,S_{H^+}(E) + f_{H^0}\,S_{H^0}(E)$$
  (Alcocer-Ávila 2021, Eqs. III.3, III.4 and III.8; the same blend as the
  reference model TILDA-V, Quinto et al. 2017).

## 4. Stopping power: Monte Carlo vs analytic (a self-consistency check)

Both curves here are computed from the *same* tables and rules: one by
random sampling, one exactly. If they agree, the sampling machinery works.
This is **not** a comparison with reference data; that comes in §5. The
figure title says so too.

The two panels show the same data with a logarithmic (left) and a linear
(right) $y$ axis. The log axis shows the whole range of energies; the
linear one shows the region around the maximum in detail.

In [ ]:
ax_log, ax_lin = plot_stopping_power_self_consistency(WATER, xs, edep_tables, eemean_tables, q=1, n_energy_points=N_SELFCONSISTENCY_ENERGY_POINTS, n_samples_per_point=N_SELFCONSISTENCY_SAMPLES_PER_POINT, seed=1, line_width=1., marker_size=1.0)
ax_log.figure.tight_layout()
save_figure(ax_log.figure, OUT_DIR, "03_stopping_power_self_consistency")
plt.show()

**Reading the figure.** The Monte Carlo points sit on the analytic line
at every energy, within their small statistical noise. The sampling of
steps and processes is correct.

The shape is the classic stopping-power curve. $S$ rises at low energy,
peaks at about 120 eV/nm near 50–60 keV (for a projectile that stays H⁺),
and then falls roughly as $1/E$ at high energy, because a fast proton
spends less time near each molecule.

The tiny, deliberate difference between the two (about 0.01–0.1%) comes
from elastic scattering. The Monte Carlo includes the small recoil energy
given to the nuclei; the analytic "electronic" stopping power leaves it out
by convention.

## 5. Stopping power: comparison with NIST PSTAR

**NIST PSTAR** is the standard reference table of proton stopping powers
(here for liquid water, material 276). The cell overlays its **electronic**
stopping power `STOP(e)`, converted from MeV·cm²/g to eV/nm by
$\times\,\rho\,[\mathrm{g/cm^3}] \times 0.1$. Electronic stopping power
excludes collisions with nuclei, as the analytic curve does.

The fixed-H⁺ curve is not the right one to compare with a real beam,
because a real beam is a mixture of H⁺ and H⁰. Three estimates of the
**charge-equilibrium** stopping power are therefore added:

- **Analytic, charge-equilibrium blended**: the closed-form blend
  $f_{H^+}S_{H^+} + f_{H^0}S_{H^0}$ (`analytic_equilibrium_stopping_power`).
- **MC, blended fixed-q campaigns**: the same weights applied to two Monte
  Carlo estimates, one for H⁺ and one for H⁰
  (`emergent_equilibrium_stopping_power`).
- **MC, single stationary-mode chain**: one long chain of collisions at fixed
  energy in which the projectile really switches between H⁺ and H⁰ through
  capture and loss events, so the charge mixture *emerges* from the
  sampling (`sample_equilibrium_stopping_power`, see §7). It is shown only
  up to 1 MeV; above that, capture is too rare for the chain to switch
  often enough.

Do not expect perfect agreement. The cross sections are theoretical and
were not adjusted to match PSTAR.

*The figure titles here and in §6 still read "self-consistency (not a
PSTAR comparison)". They come from the plotting function reused from §4
and describe only its two original curves, not the reference data
overlaid on top.*

In [ ]:
# NIST PSTAR, protons in liquid water (material 276), electronic stopping power STOP(e).
# Fetched live 2026-08-11 from https://physics.nist.gov/cgi-bin/Star/ap_table-t.pl
# (matno=276, ShowDefault=on, prog=PSTAR) -- default energy grid, MeV cm^2/g.
PSTAR_E_MeV = np.array([
    0.008, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0,
    2.0, 5.0, 10.0, 20.0, 50.0, 100.0, 200.0, 300.0,
])
PSTAR_STOP_E_MeV_cm2_g = np.array([
    378.2, 422.9, 567.3, 774.0, 814.5, 660.4, 412.8, 260.6,
    158.5, 79.06, 45.64, 26.05, 12.44, 7.286, 4.491, 3.519,
])
PSTAR_E_eV = PSTAR_E_MeV * 1.0e6
PSTAR_S_eV_per_nm = PSTAR_STOP_E_MeV_cm2_g * WATER.density_g_cm3 * 0.1  # MeV cm^2/g -> eV/nm

axes = plot_stopping_power_self_consistency(WATER, xs, edep_tables, eemean_tables, q=1,
                                             n_energy_points=N_SP_ENERGY_POINTS, n_samples_per_point=N_SP_SAMPLES_PER_POINT, seed=1,
                                             line_width=1.0, marker_size=2.0)

# Charge-state-equilibrium blend (Eq. III.8), on the same E_grid the self-consistency call above used.
e_min, e_max = xs.energy_range(1)
E_grid = np.geomspace(e_min, e_max, N_SP_ENERGY_POINTS)
S_eff = [analytic_equilibrium_stopping_power(WATER, xs, E, edep_tables, eemean_tables) for E in E_grid]

# Two MC-emergent counterparts to the closed-form blend above (see §7's
# "Validating the equilibrium charge fractions..." for the underlying
# stationary-mode-chain machinery).
sp_mc_engine = TrackEngine(WATER, xs, edep_tables=edep_tables, eemean_tables=eemean_tables, seed=1)

# (A) Blend two independently-sampled fixed-q emergent_stopping_power
# campaigns by the SAME closed-form weight -- practical at every energy,
# since it needs no charge-flip statistics, only per-q sampling.
S_eff_blend_mc = [emergent_equilibrium_stopping_power(sp_mc_engine, E, n_samples=N_SP_SAMPLES_PER_POINT) for E in E_grid]

# (B) A single stationary-mode chain per point, where the charge mix
# genuinely emerges from real CAPTURE/LOSS events -- only practical on the
# grid's low-energy points, where capture isn't vanishingly rare (§7).
E_grid_chain = E_grid[E_grid <= 1.0e6]
S_eff_chain_mc = []
for E in E_grid_chain:
    chain_engine = TrackEngine(WATER, xs, edep_tables=edep_tables, eemean_tables=eemean_tables, seed=1)
    S_eff_chain_mc.append(sample_equilibrium_stopping_power(chain_engine, E, q0=1, n_collisions=N_CHAIN_COLLISIONS))

# Two panels (log/linear y-scale, see plot_stopping_power_self_consistency's
# docstring) -- overlay every extra series onto both so they stay in sync.
for ax in axes:
    ax.plot(PSTAR_E_eV, PSTAR_S_eV_per_nm, "s--", color="black", markersize=2, label="NIST PSTAR (electronic, water)")
    ax.plot(E_grid, S_eff, "o-", color="tab:green", markersize=2, linewidth=1.0,
            label="Analytic, charge-equilibrium blended (Eq. III.8)")
    ax.plot(E_grid, S_eff_blend_mc, "x", color="tab:orange", markersize=6,
            label="MC, blended fixed-q campaigns")
    ax.plot(E_grid_chain, S_eff_chain_mc, "^", color="tab:purple", markersize=5,
            label="MC, single stationary-mode chain")
    legend_position = "lower left" if ax.get_yscale() == "log" else "upper right"
    ax.legend(fontsize=5, loc=legend_position, framealpha=1.0)
axes[0].figure.tight_layout()
save_figure(axes[0].figure, OUT_DIR, "04_stopping_power_vs_pstar")
plt.show()

**Reading the figure.**

- **Above about 300 keV all curves agree** with PSTAR. At high energy
  capture is negligible and the theory used for ionization is accurate.
- **Near the maximum the model is higher than PSTAR.** The fixed-H⁺
  analytic curve peaks at about 120 eV/nm, PSTAR at about 81 eV/nm.
- **Including the charge-state equilibrium (green) reduces the gap**, to
  a peak of about 106 eV/nm near 70 keV. H⁰ loses less energy than H⁺,
  and at low energy the projectile spends much of its path as H⁰.
- The three equilibrium estimates agree with one another, so the residual
  gap to PSTAR is a property of the input cross sections, not of the
  sampling. §6 shows where it comes from.

## 6. Comparison with experimental stopping-power data

The same axes again, now with published measurements of proton stopping
power in water (Baek 2006, Mitterschiffthaler 1990, Phillips 1953,
Reynolds 1953, Siiskonen 2011) and with the reference model of Quinto et
al. 2017. The data are in `data_comparison/stopping_power_water/` (references in its README), in keV/u and keV/µm:
energies are multiplied by 1000 to get eV, and keV/µm equals eV/nm.

`SP_2017_Quinto_EJPD.dat` is not a measurement. It is the TILDA-V
reference model whose cross sections this code uses, so comparing with it
is a theory-to-theory check.

In [ ]:
SP_COMPARE_DIR = Path("..") / "data_comparison" / "stopping_power_water"
SP_COMPARE_FILES = {
    "Baek 2006": "Exp_SP_Baek_2006.dat",
    "Mitterschiffthaler 1990": "Exp_SP_Mitterschiffthaler_1990.dat",
    "Phillips 1953": "Exp_SP_Phillips_1953.dat",
    "Reynolds 1953": "Exp_SP_Reynolds_1953.dat",
    "Siiskonen 2011": "Exp_SP_Siiskonen_2011.dat",
}

axes = plot_stopping_power_self_consistency(WATER, xs, edep_tables, eemean_tables, q=1,
                                             n_energy_points=N_SP_ENERGY_POINTS, n_samples_per_point=N_SP_SAMPLES_PER_POINT, seed=1,
                                             line_width=1., marker_size=2.0)

# Load each experimental dataset once, up front.
sp_compare_data = {}
for label, filename in SP_COMPARE_FILES.items():
    E_keV_u, SP_keV_um, err_keV_um = np.loadtxt(SP_COMPARE_DIR / filename, unpack=True)
    # keV/u reading (A=1 exactly for protons, so x1000 -> eV needs no separate mass factor, unlike E_BEAM_KEV_U in notebook 02).
    E_eV = E_keV_u * 1000.0
    SP_eV_per_nm = SP_keV_um  # keV/um -> eV/nm is exact (x1)
    err_eV_per_nm = err_keV_um
    sp_compare_data[label] = (E_eV, SP_eV_per_nm, err_eV_per_nm)

E_keV_u_q, SP_keV_um_q = np.loadtxt(SP_COMPARE_DIR / "SP_2017_Quinto_EJPD.dat", unpack=True)
E_eV_q = E_keV_u_q * 1000.0

# Two panels (log/linear y-scale) -- overlay every extra series onto both.
for ax in axes:
    ax.plot(PSTAR_E_eV, PSTAR_S_eV_per_nm, "s--", color="black", markersize=2, label="NIST PSTAR (electronic, water)")
    ax.plot(E_grid, S_eff, "d-", color="tab:green", markersize=2, linewidth=1.0,
            label="Analytic, charge-equilibrium blended (Eq. III.8)")

    for label, (E_eV, SP_eV_per_nm, err_eV_per_nm) in sp_compare_data.items():
        ax.errorbar(E_eV, SP_eV_per_nm, yerr=err_eV_per_nm, fmt="o", markersize=2.0, linewidth=.5, label=label)

    ax.plot(E_eV_q, SP_keV_um_q, "^", markersize=2.0, color="black",
            label="Quinto et al. 2017 (reference model)")

    legend_position = "lower left" if ax.get_yscale() == "log" else "upper right"
    ax.legend(fontsize=5, loc=legend_position, framealpha=1.0)
axes[0].figure.tight_layout()
save_figure(axes[0].figure, OUT_DIR, "05_stopping_power_vs_experiment")
plt.show()

**Reading the figure.** The measurements (coloured points with error
bars) lie **between** PSTAR and the model near the maximum, and closer to
the model. The charge-equilibrium curve (green) follows them over most of
the range; at the maximum it is roughly 10–15% above them. The fixed-H⁺
curve is about 25% too high there: charge exchange is not a small
correction at these energies. The Quinto et al. reference model (black
triangles) sits close to the measurements, as expected for the model this
code reproduces.

Baek 2006 extends below 8 keV, outside the tabulated range, so the
code has nothing to compare there.

### Residual check: is the remaining gap a vapour-vs-liquid effect?

The cross sections used here were computed for water **vapour**, not liquid
water (as in TILDA-V; Alcocer-Ávila 2021, Ch. III). The stopping power of
the two phases differs: according to NIST, by 13.8% at 10 keV and 12.6% at
100 keV, falling below 1% above 1 MeV. The measurements also come in both
phases:

- **vapour**: Reynolds, Phillips, Mitterschiffthaler, Baek;
- **liquid**: Siiskonen and NIST PSTAR.

If the remaining gap is mainly a phase effect, the model should agree
better with the **vapour** data and overshoot the **liquid** data. The cell
below tests this. For each measured point it computes the relative
residual $(S_{\text{sim}} - S_{\text{exp}})/S_{\text{exp}}$, evaluating the
charge-equilibrium curve at that point's exact energy, and averages the
residuals per phase. Points below the 8 keV table floor are skipped.

In [ ]:
# Phase classification per Alcocer-Avila 2021, Ch. III (pp. 96-98).
DATASET_PHASE = {
    "Baek 2006": "vapor",
    "Mitterschiffthaler 1990": "vapor",
    "Phillips 1953": "vapor",
    "Reynolds 1953": "vapor",
    "Siiskonen 2011": "liquid",
}

e_min, e_max = xs.energy_range(1)

def relative_residuals(E_eV, S_exp):
    """(S_sim - S_exp) / S_exp at each point, evaluating the charge-
    equilibrium-blended curve exactly at each experimental energy. Points
    outside the tabulated cross-section range are skipped (same
    no-extrapolation policy as the loader)."""
    residuals = []
    for E, S in zip(E_eV, S_exp):
        if not (e_min <= E <= e_max):
            continue
        S_sim = analytic_equilibrium_stopping_power(WATER, xs, E, edep_tables, eemean_tables)
        residuals.append((S_sim - S) / S)
    return np.array(residuals)

vapor_residuals = []
liquid_residuals = []
for label, (E_eV, SP_eV_per_nm, _err) in sp_compare_data.items():
    r = relative_residuals(E_eV, SP_eV_per_nm)
    phase = DATASET_PHASE[label]
    (vapor_residuals if phase == "vapor" else liquid_residuals).extend(r)
    n_skipped = len(E_eV) - len(r)
    skip_note = f", {n_skipped} skipped (below 8 keV floor)" if n_skipped else ""
    print(f"{label:28s} ({phase:6s}): n={len(r):2d}{skip_note}  mean residual = {r.mean():+.1%}")

pstar_residuals = relative_residuals(PSTAR_E_eV, PSTAR_S_eV_per_nm)
liquid_residuals.extend(pstar_residuals)
print(f"{'NIST PSTAR':28s} ({'liquid':6s}): n={len(pstar_residuals):2d}  mean residual = {pstar_residuals.mean():+.1%}")

vapor_residuals = np.array(vapor_residuals)
liquid_residuals = np.array(liquid_residuals)
print()
print(f"Pooled vapor-phase datasets            (n={len(vapor_residuals):2d}): mean residual = {vapor_residuals.mean():+.1%}")
print(f"Pooled liquid-phase datasets incl PSTAR (n={len(liquid_residuals):2d}): mean residual = {liquid_residuals.mean():+.1%}")

**Reading the result.** The answer is **no**: the simple phase
explanation does not hold. The model is on average **8% above the vapour**
measurements, the very phase its cross sections were computed for. It is
only **0.4% above the liquid** data on average, although this average
hides a spread from −6.7% (Siiskonen) to +3.9% (PSTAR).

So the remaining overshoot near the maximum is not explained by the phase
of the target. It most likely comes from the input cross sections
themselves at low energy, where the theoretical models are least accurate.
Note that the averages mix points from very different energies, and most
of the vapour points lie near the maximum, where the gap is largest. Treat
this as a first diagnosis, not a final one.

## 7. Checking the charge-state equilibrium against the sampling

§5–6 used the closed-form fractions $f_{H^+} = \sigma_L/(\sigma_L+\sigma_C)$
and $f_{H^0} = \sigma_C/(\sigma_L+\sigma_C)$. Does the transport machinery
really produce them, when it picks capture and loss events at random and
switches `q` accordingly?

`sample_equilibrium_charge_fractions` checks this directly. It runs a chain
of 30 000 collisions at a **fixed** energy $E$ (no energy loss between
collisions: the "stationary mode" of Alcocer-Ávila's thesis). The
projectile starts as H⁺ and changes charge whenever a capture or loss event
is drawn. The fraction of the *path length* spent as H⁺ is then compared
with the closed form. Path length, not number of collisions, is the right
weight: H⁺ and H⁰ have different mean free paths.

**Only practical up to about 1 MeV.** Capture becomes very rare at high
energy. Its probability per H⁺ collision is about 6% at 100 keV/u, 0.09% at
500 keV/u and 0.002% at 2000 keV/u, so a chain of 30 000 collisions would
see too few switches to settle. The probe energies are 10–1000 keV/u.

In [ ]:
CHAIN_PROBE_E_KEV_U = np.array([10.0, 20.0, 50.0, 100.0, 300.0, 1000.0])  # 8 keV/u is the tabulated floor for CAPTURE -- stay above it

# Closed-form curve (Eqs. III.3/III.4), fine grid.
E_dense = np.geomspace(xs.energy_range(1)[0], 2.0e6, 200)
f_Hplus_closed_dense = np.array([equilibrium_charge_fractions(xs, E)[0] for E in E_dense])

# Empirical (MC) fractions at each probe energy, one fresh stationary-mode
# chain per point, plus the closed-form value at that exact energy for a
# direct numeric comparison.
probe_E_eV = CHAIN_PROBE_E_KEV_U * 1.0e3 * constants.PROTON_NUCLEON_COUNT
f_Hplus_closed_probe = np.array([equilibrium_charge_fractions(xs, E)[0] for E in probe_E_eV])
f_Hplus_mc = []
for E in probe_E_eV:
    chain_engine = TrackEngine(WATER, xs, edep_tables=edep_tables, eemean_tables=eemean_tables, seed=1)
    f_Hplus, _f_H0 = sample_equilibrium_charge_fractions(chain_engine, E, q0=1, n_collisions=N_CHAIN_COLLISIONS)
    f_Hplus_mc.append(f_Hplus)
f_Hplus_mc = np.array(f_Hplus_mc)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(E_dense, f_Hplus_closed_dense, "-", color="tab:blue", label="f_H+ (Eq. III.3, closed form)")
ax.plot(E_dense, 1.0 - f_Hplus_closed_dense, "-", color="tab:red", label="f_H0 (Eq. III.4, closed form)")
ax.plot(probe_E_eV, f_Hplus_mc, "o", color="tab:blue", markersize=5,
        label="f_H+ (MC, stationary-mode chain)")
ax.plot(probe_E_eV, 1.0 - f_Hplus_mc, "o", color="tab:red", markersize=5,
        label="f_H0 (MC, stationary-mode chain)")
ax.set_xscale("log")
ax.set_xlabel("Energy [eV]")
ax.set_ylabel("Equilibrium charge-state fraction")
ax.legend(fontsize=6)
save_figure(fig, OUT_DIR, "06_equilibrium_charge_fractions")
plt.show()

for E_kev, f_closed, f_mc in zip(CHAIN_PROBE_E_KEV_U, f_Hplus_closed_probe, f_Hplus_mc):
    print(f"E={E_kev:>7.1f} keV/u  f_H+: closed-form={f_closed:.4f}  MC={f_mc:.4f}  |diff|={abs(f_closed - f_mc):.4f}")

**Reading the result.** The sampled fractions (points) fall on the
closed-form curves at every probe energy; the printed differences are at
most 0.003. The random capture/loss switching therefore reproduces the
equilibrium exactly as the balance equation predicts.

Physically: below about 50 keV/u the projectile is **mostly neutral**
($f_{H^+} = 0.12$ at 10 keV/u, 0.23 at 20 keV/u). At 300 keV/u it is a bare
proton about 99% of the time. The crossover ($f_{H^+} = 0.5$, near
43 keV/u) lies just below the stopping-power maximum (60–70 keV), which is
why charge exchange matters so much there.

## 8. Bragg curve: dose vs depth for one track

A **depth-dose scorer** is attached to a new engine. It receives every
energy deposit as it happens and adds it to one of 80 depth bins
(0–8.5 µm). Its transverse size is huge (±1 mm) so that nothing is lost
sideways. Dose in each bin is energy / mass of water in the bin.

This is a single 500 keV/u proton (with `seed=7`, the same track as §3), so
the curve is noisy. Notebook 02 does the same with a beam of 500 protons.

In [ ]:
z_edges = np.linspace(0.0, 8500.0, 81)
bragg_scorer = make_bragg_curve_scorer(WATER, z_edges=z_edges, transverse_half_width_nm=1.0e6)
engine2 = TrackEngine(WATER, xs, edep_tables=edep_tables, eemean_tables=eemean_tables, scorers=[bragg_scorer], seed=7)
primary2 = Particle(kind="proton", position=np.zeros(3), direction=np.array([0.0, 0.0, 1.0]),
                     energy=E_probe, q=1)
engine2.transport(primary2)
ax = plot_bragg_curve(bragg_scorer, line_width=1.)
save_figure(ax.figure, OUT_DIR, "07_bragg_curve_single_track")
plt.show()

**Reading the figure.**

- The dose **rises with depth** from about $1.7\times10^{-6}$ Gy near the
  entrance to a **peak of about $4.4\times10^{-6}$ Gy near 6.9 µm**, then
  drops to zero at about 7.6 µm, the end of the track. This is the
  **Bragg peak**: as the proton slows down its stopping power rises
  (§4–5), so it deposits more energy per nm near the end.
- The drop is very sharp because this is a single track. For a beam, the
  end of the track varies from proton to proton (range straggling) and the
  fall-off is smoother.
- The bin-to-bin jitter is the randomness of a single track; it averages
  out over many protons.
- The absolute dose is tiny because one proton is spread over a
  2 mm × 2 mm bin. Only the shape matters here.

## 9. LET: track-averaged and dose-averaged

**LET** (linear energy transfer) is the energy the projectile loses per
unit path length: the stopping power seen *along a track*. It changes as
the proton slows down, so a track is summarised by averages. The engine
records every collision of the primary proton as a pair (step length
$s_i$, energy lost $\Delta E_i$) in `engine.collision_log`, here for the
track of §3.

- **Track-averaged LET**, $\bar L_T = \sum_i \Delta E_i / \sum_i s_i$: total
  energy lost over total path. Every nm of track counts equally.
- **Dose-averaged LET**, $\bar L_D$: each part of the track is weighted by
  the energy it deposits, so the high-LET end counts more. It is always
  $\ge \bar L_T$, and it is the average usually related to biological
  effect.

**$\bar L_D$ needs a length scale.** Cut the track into segments of length
$\Delta$, let $E_k$ be the energy lost in segment $k$ and $\ell_k$ its
length, so that its LET is $L_k = E_k/\ell_k$. Then

$$\bar L_D(\Delta) = \frac{\sum_k L_k E_k}{\sum_k E_k} = \frac{\sum_k E_k^2/\ell_k}{\sum_k E_k}.$$

Why not use single collisions, $L_i = \Delta E_i / s_i$? Because the step
lengths $s_i$ are exponentially distributed, and the mean of $1/s$ for an
exponential distribution is infinite: rare, very short steps would dominate
and the average would never settle. A fixed $\Delta$ avoids that, but makes
the result depend on $\Delta$. We use $\Delta = 100$ nm, much longer than
the mean free path (about 1 nm) and much shorter than the track (7.6 µm).
The cell also scans $\Delta$ to show the dependence (see
`ion_track_mc/scoring/README_scoring.md` §3).

In [ ]:
LET_SEGMENT_NM = 100.0  # Delta: >> mean free path (~nm), << track length

print(f"track-averaged LET = {track_averaged_LET(engine.collision_log):.3f} eV/nm (= keV/um)")
print(f"dose-averaged LET  = {dose_averaged_LET(engine.collision_log, segment_length_nm=LET_SEGMENT_NM):.3f} eV/nm "
      f"(= keV/um), Delta = {LET_SEGMENT_NM:g} nm")

track_length_nm = sum(s for s, _ in engine.collision_log)
print(f"\ntrack length = {track_length_nm:.0f} nm; L_D vs segment length Delta:")
for delta in [10.0, 30.0, 100.0, 300.0, 1000.0, track_length_nm]:
    print(f"  Delta = {delta:8.0f} nm -> L_D = {dose_averaged_LET(engine.collision_log, segment_length_nm=delta):.3f} eV/nm")

**Reading the result.**

- $\bar L_T = 64.4$ keV/µm: 492 keV lost over 7.6 µm.
- $\bar L_D = 70.6$ keV/µm at $\Delta = 100$ nm, higher than $\bar L_T$ as
  it must be, because the dense end of the track carries extra weight.
- **The scan shows the scale dependence.** $\bar L_D$ grows slowly as
  $\Delta$ shrinks (75.0 keV/µm at 10 nm): small segments resolve the
  fluctuations of the energy loss, and big local values get bigger
  weights. When $\Delta$ equals the whole track length there is only one
  segment, and $\bar L_D$ becomes exactly $\bar L_T$ (64.4).
- The lesson: a dose-averaged LET is only meaningful together with the
  scale it was measured at. This matters when LET values from different
  codes or experiments are compared.

## Summary and limitations

What this notebook showed, step by step:

1. Steps between collisions follow the exponential law with the correct
   mean free path (§2).
2. One track conserves energy exactly (§3).
3. Monte Carlo stopping power reproduces the exact expectation of the
   code's rules (§4).
4. Including the H⁺ ↔ H⁰ charge-state equilibrium is essential near the
   stopping-power maximum. With it, the model follows the measurements
   above about 300 keV and is about 10–15% high near the maximum (§5–6).
   That gap is not a simple vapour-vs-liquid effect (§6).
5. The sampled charge switching reproduces the equilibrium fractions
   (§7).
6. A single track has a Bragg peak at its end (§8), and its LET depends on
   how it is averaged and at what scale (§9).

Limitations to keep in mind when using these numbers: δ-electrons deposit
their energy on the spot, liquid water is modelled with vapour-phase cross
sections, tracks stop at 8 keV, and lateral scattering has not been
validated against reference data. **Next: notebook 02**, a beam of many
protons and the dose it deposits.